In [0]:
import boto3
from pyspark.sql.functions import lit, col , current_timestamp


access_key = dbutils.secrets.get(scope="Aws_secrets_retail", key="aws-access-key-id")
secret_key = dbutils.secrets.get(scope="Aws_secrets_retail", key="aws-secret-access-key")

bucket_name = "we-retail-project"
catalog = "hive_metastore"
database = "we_retail_hub"

s3 = boto3.client(
    's3',
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name='us-east-1'
)

# Configuration for each entity
entities = [
    {
        "prefix": "we_retail_hub/Full_load/campaign/",
        "id_col": "campaign_id",
        "id_prefix": "CAMP",
        "dup_cols": ["campaign_id", "campaign_name"],
        "error_msg": "Invalid campaign_id format or duplicate campaign names found."
    },
    {
        "prefix": "we_retail_hub/Full_load/customer/",
        "id_col": "customer_id",
        "id_prefix": "CUST",
        "dup_cols": ["first_name", "last_name", "email", "customer_id"],
        "error_msg": "Invalid customer_id format or duplicate customer names found."
    },
    {
        "prefix": "we_retail_hub/Full_load/product/",
        "id_col": "product_id",
        "id_prefix": "PROD",
        "dup_cols": ["product_id", "product_name", "category", "brand", "origin_location"],
        "error_msg": "Invalid product_id format or duplicate product names found."
    },
    {
        "prefix": "we_retail_hub/Full_load/salesperson/",
        "id_col": "salesperson_id",
        "id_prefix": "SP",
        "dup_cols": ["salesperson_id", "salesperson_name", "salesperson_role"],
        "error_msg": "Invalid salesperson_id format or duplicate salesperson names found."
    },
    {
        "prefix": "we_retail_hub/Full_load/store/",
        "id_col": "store_id",
        "id_prefix": "STR",
        "dup_cols": ["store_id", "store_name", "store_type", "store_location"],
        "error_msg": "Invalid store_id format or duplicate store names found."
    },
]

audit_columns = {"aud_load_timestamp", "aud_file_name"}

for entity in entities:
    prefix = entity["prefix"]
    paginator = s3.get_paginator('list_objects_v2')
    files = [
        obj['Key']
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix)
        for obj in page.get('Contents', [])
        if not obj['Key'].endswith('/')
    ]

    if len(files) == 0:
        print(f"No new files to process for {prefix}")
        continue

    for s3_key in files:
        table_name = s3_key.split('/')[-1].replace('.csv', '')
        table_name = table_name.split('_HL')[0]

        try:
            full_path = f"s3://{bucket_name}/{s3_key}"
            df = spark.read.option("header", "true").option("inferSchema", "true").csv(full_path)

            # Generate SHA hash for dup_cols and add as row_key
            df = df.withColumn(
                "row_key",
                sha2(concat_ws("||", *entity["dup_cols"]), 256)
            )
            # Validate ID prefix
            invalid_count = df.filter(~col(entity["id_col"]).startswith(entity["id_prefix"])).count()

            # Check duplicates
            dup_cols = [col(c) for c in entity["dup_cols"]]
            duplicate_count = df.groupBy(*dup_cols).count().filter(col("count") > 1).count()

            # Check column names with catalog schema, excluding audit columns
            table_schema = spark.table(f"{catalog}.{database}.{table_name}").schema
            table_columns = set(field.name for field in table_schema if field.name not in audit_columns)
            df_columns = set(col for col in df.columns if col not in audit_columns)
            if table_columns != df_columns:
                raise ValueError(f"Column mismatch: Table columns {table_columns}, DataFrame columns {df_columns}")

            if invalid_count > 0 or duplicate_count > 0:
                raise ValueError(entity["error_msg"])

            # Add audit columns (overwrite if already present)
            df = df.drop(*[c for c in audit_columns if c in df.columns])

            df = df.withColumn("aud_load_timestamp", current_timestamp())\
                .withColumn("aud_upd_timestamp",current_timestamp())\
                .withColumn("active_flag","Y")\
                .withColumn("aud_file_name", lit(full_path))

            # df.display()

            df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{database}.{table_name}")

            print(f"Load Successful for {prefix}")
            s3.delete_object(Bucket=bucket_name, Key=s3_key)
        except ValueError as e:
            print(f"Data Validation checks failed for \n {s3_key}: \n{e}")
            s3.delete_object(Bucket=bucket_name, Key=s3_key)
        except Exception as e:
            print(f"Unexpected error processing {s3_key}: {e}")
            s3.delete_object(Bucket=bucket_name, Key=s3_key)